In [30]:
import pandas as pd
import numpy as np

In [31]:
df=pd.read_csv("IMDB%20Dataset.csv")
df=df.iloc[:10000]

In [32]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [33]:
df["review"][1]

'A wonderful little production. <br /><br />The filming technique is very unassuming- very old-time-BBC fashion and gives a comforting, and sometimes discomforting, sense of realism to the entire piece. <br /><br />The actors are extremely well chosen- Michael Sheen not only "has got all the polari" but he has all the voices down pat too! You can truly see the seamless editing guided by the references to Williams\' diary entries, not only is it well worth the watching but it is a terrificly written and performed piece. A masterful production about one of the great master\'s of comedy and his life. <br /><br />The realism really comes home with the little things: the fantasy of the guard which, rather than use the traditional \'dream\' techniques remains solid then disappears. It plays on our knowledge and our senses, particularly with the scenes concerning Orton and Halliwell and the sets (particularly of their flat with Halliwell\'s murals decorating every surface) are terribly well d

In [34]:
df["sentiment"].value_counts()

sentiment
positive    5028
negative    4972
Name: count, dtype: int64

In [35]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [36]:
df.duplicated().sum()

np.int64(17)

In [37]:
df.drop_duplicates(inplace=True)

In [38]:
df.duplicated().sum()

np.int64(0)

Basic Preprocessing
Remove tags
LowerCase
remove stopwds

In [39]:
#HTML  tag remove
import re
def remove_tag(raw_text):
    cleand_text=re.sub(re.compile("<.*?>"),"",raw_text)
    return cleand_text

In [40]:
df["review"]=df["review"].apply(remove_tag)
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. The filming tec...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
9995,"Fun, entertaining movie about WWII German spy ...",positive
9996,Give me a break. How can anyone say that this ...,negative
9997,This movie is a bad movie. But after watching ...,negative
9998,This is a movie that was probably made to ente...,negative


In [41]:
df["review"] = df["review"].apply(lambda x: x.lower())

In [42]:
from nltk.corpus import stopwords

sw_list = stopwords.words("english")

df["review"] = df["review"].apply(
    lambda x: [item for item in x.split() if item not in sw_list]
).apply(
    lambda x: " ".join(x)
)

In [43]:
df

,review,sentiment
0,one reviewers mentioned watching 1 oz episode ...,positive
1,wonderful little production. filming technique...,positive
2,thought wonderful way spend time hot summer we...,positive
3,basically there's family little boy (jake) thi...,negative
4,"petter mattei's ""love time money"" visually stu...",positive
...,...,...
9995,"fun, entertaining movie wwii german spy (julie...",positive
9996,"give break. anyone say ""good hockey movie""? kn...",negative
9997,movie bad movie. watching endless series bad h...,negative
9998,"movie probably made entertain middle school, e...",negative


In [44]:
X=df.iloc[:,0:1]
y=df["sentiment"]

In [45]:
X

,review
0,one reviewers mentioned watching 1 oz episode ...
1,wonderful little production. filming technique...
2,thought wonderful way spend time hot summer we...
3,basically there's family little boy (jake) thi...
4,"petter mattei's ""love time money"" visually stu..."
...,...
9995,"fun, entertaining movie wwii german spy (julie..."
9996,"give break. anyone say ""good hockey movie""? kn..."
9997,movie bad movie. watching endless series bad h...
9998,"movie probably made entertain middle school, e..."


In [46]:
y

0       positive
1       positive
2       positive
3       negative
4       positive
          ...   
9995    positive
9996    negative
9997    negative
9998    negative
9999    positive
Name: sentiment, Length: 9983, dtype: object

In [47]:
from sklearn.preprocessing import LabelEncoder
en=LabelEncoder()
y=en.fit_transform(y)

In [48]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [49]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)

In [50]:
X_train.shape

(7986, 1)

In [58]:
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(
    max_features=5000,
    min_df=2,
    dtype='int32'
)

In [59]:
X_train_bow=cv.fit_transform(X_train["review"]).toarray()
X_test_bow=cv.transform(X_test["review"]).toarray()

In [60]:
from sklearn.naive_bayes import GaussianNB
gnb=GaussianNB()
gnb.fit(X_train_bow,y_train)

,"priors priors: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
,"var_smoothing var_smoothing: float, default=1e-9Portion of the largest variance of all features that is added tovariances for calculation stability... versionadded:: 0.20",1e-09


In [62]:
y_prd=gnb.predict(X_test_bow)
from sklearn.metrics import accuracy_score,confusion_matrix
accuracy_score(y_test,y_prd)

0.7646469704556835

In [64]:
confusion_matrix(y_test,y_prd)

array([[794, 158],
       [312, 733]])

In [67]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)
rf.fit(X_train_bow, y_train)
y_pred = rf.predict(X_test_bow)
accuracy_score(y_test, y_pred)

0.8382573860791187

In [68]:
cv=CountVectorizer(max_features=3000)
X_train_bow=cv.fit_transform(X_train["review"]).toarray()
X_test_bow=cv.transform(X_test["review"]).toarray()
rf=RandomForestClassifier()
rf.fit(X_train_bow,y_train)
y_prd=rf.predict(X_test_bow)
accuracy_score(y_test,y_prd)

0.8452679018527791

In [70]:
cv=CountVectorizer(ngram_range=(1,3),max_features=5000)
X_train_bow=cv.fit_transform(X_train["review"]).toarray()
X_test_bow=cv.transform(X_test["review"]).toarray()
rf=RandomForestClassifier()
rf.fit(X_train_bow,y_train)
y_prd=rf.predict(X_test_bow)
accuracy_score(y_test,y_prd)

0.8407611417125689

In [73]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer(max_features=2000)
X_train_tfidf=tfidf.fit_transform(X_train["review"]).toarray()
X_test_tfidf=tfidf.transform(X_test["review"])
rf=RandomForestClassifier()
rf.fit(X_train_bow,y_train)
y_prd=rf.predict(X_test_bow)
accuracy_score(y_test,y_prd)

0.8442663995993991

In [81]:
import gensim
from nltk import sent_tokenize
from gensim.utils import simple_preprocess
strory=[]
for doc in df["review"]:
    raw_sent=sent_tokenize(doc)
    for sent in raw_sent:
        strory.append(simple_preprocess(sent))

In [82]:
from gensim.models import Word2Vec
model = Word2Vec(window=10, min_count=2)

In [83]:
model.build_vocab(strory)

In [85]:
model.train(strory,total_examples=model.corpus_count,epochs=model.epochs)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


(5850091, 6186875)

In [86]:
len(model.wv.index_to_key)

31845

In [87]:
def document_vector(doc):
    doc=[word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc],axis=0)

In [88]:
document_vector(df["review"].values[0])

array([-0.13982207,  0.398345  ,  0.15073295, -0.11710634, -0.0048352 ,
       -0.513223  ,  0.05053427,  0.86656326, -0.2557317 , -0.06304176,
       -0.09867161, -0.5388082 ,  0.02235334,  0.23475803,  0.05900026,
       -0.18010713,  0.1432847 , -0.4827698 ,  0.0692834 , -0.68342364,
        0.12650727,  0.12124205,  0.14827897, -0.13643746, -0.18935944,
       -0.11475944, -0.2155709 , -0.21938512, -0.27497151, -0.01383011,
        0.4300362 , -0.00540981,  0.03010582, -0.26935405, -0.30974072,
        0.31131858,  0.14215243, -0.42155254, -0.21806791, -0.6747343 ,
        0.15122034, -0.25382113, -0.25026977, -0.04906467,  0.39232582,
       -0.17418092, -0.49403337, -0.0951944 ,  0.14994234,  0.28477293,
        0.17051017, -0.29007897, -0.24408045, -0.08639473, -0.364708  ,
        0.0518624 ,  0.4014261 ,  0.08373741, -0.26663244,  0.04468681,
       -0.04556951,  0.09069022, -0.06589883,  0.0154596 , -0.4006392 ,
        0.32660347,  0.01771863,  0.2264575 , -0.6357681 ,  0.24

In [99]:
from tqdm import tqdm
X = []
for doc in df["review"].values: 
    X.append(document_vector(doc))

In [100]:
X=np.array(X)

In [101]:
X[0]

array([-0.13982207,  0.398345  ,  0.15073295, -0.11710634, -0.0048352 ,
       -0.513223  ,  0.05053427,  0.86656326, -0.2557317 , -0.06304176,
       -0.09867161, -0.5388082 ,  0.02235334,  0.23475803,  0.05900026,
       -0.18010713,  0.1432847 , -0.4827698 ,  0.0692834 , -0.68342364,
        0.12650727,  0.12124205,  0.14827897, -0.13643746, -0.18935944,
       -0.11475944, -0.2155709 , -0.21938512, -0.27497151, -0.01383011,
        0.4300362 , -0.00540981,  0.03010582, -0.26935405, -0.30974072,
        0.31131858,  0.14215243, -0.42155254, -0.21806791, -0.6747343 ,
        0.15122034, -0.25382113, -0.25026977, -0.04906467,  0.39232582,
       -0.17418092, -0.49403337, -0.0951944 ,  0.14994234,  0.28477293,
        0.17051017, -0.29007897, -0.24408045, -0.08639473, -0.364708  ,
        0.0518624 ,  0.4014261 ,  0.08373741, -0.26663244,  0.04468681,
       -0.04556951,  0.09069022, -0.06589883,  0.0154596 , -0.4006392 ,
        0.32660347,  0.01771863,  0.2264575 , -0.6357681 ,  0.24

In [102]:
from sklearn.preprocessing import LabelEncoder
en=LabelEncoder()
y=en.fit_transform(df["sentiment"])

In [103]:
y

array([1, 1, 1, ..., 0, 0, 1], shape=(9983,))

In [105]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
rf=RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test, y_pred)

ValueError: could not convert string to float: 'waiting superhero movie like long time. "mystery men" takes place among classic comic-strip spoofs tv like "batman" "captain nice" cartoons like "underdog" "super chicken." spirit lives them: comic tongue-in-cheek tone; courage aim heroic life risk looking ridiculous; not-so-sure-footed way characters manage prevail adversaries. misfired spark nobility igniting weak ordinary, wonderful see glow high bright here."mystery men" opens party nursing home. wish kinka usher sense give energy life old people scene. is, looks like something george romero might devised. need get feeling old people sharp everyone else, feels patronizing. time red eyes crash festivities, half expect tom waits plays weapons inventor penchant ladies eighties stand shout: "just party needs--a little excitement!" writer neil cuthbert sense, would waits mixing intruders egging partiers same. would made rousing beginning, better introduction troublesome trio: shoveler (wil